# 07 - LoRA training (Colab dry-run)


## Goal

Show the resolved training config under `profile=colab_t4` and print the `accelerate launch` command a learner *would* run. This notebook does **not** import `torch` and does **not** train anything.


## Prerequisites

- `configs/training.yaml` and `configs/models.yaml` present (they are committed).
- PyYAML installed (it is a base dep of the lab).
- No GPU, no model weights, no internet required.


## Environment bootstrap

This cell makes the notebook portable between a local checkout and Colab.

- **Local**: when the notebook lives inside the repo, we add the repo root to `sys.path`
  so the `src` package imports cleanly.
- **Colab**: the import will fail with `ModuleNotFoundError`. We catch that and print a
  one-line reminder showing the `git clone` the learner should run. We deliberately do
  **not** execute the clone for them — the lab policy is *recipes only, no auto-downloads*.


In [ ]:
import sys
from pathlib import Path

try:
    # Local checkout: walk up from the notebook to the repo root.
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "common" / "paths.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break
    from src.common.paths import REPO_ROOT, MINI_REPO, ensure_dirs
    ensure_dirs()
    print(f"repo root: {REPO_ROOT}")
    print(f"mini repo: {MINI_REPO}")
except ModuleNotFoundError:
    print("`src` not importable. If you are on Colab, run this in a separate cell:")
    print("    !git clone https://example.invalid/kde_ontology_slm_lab.git")
    print("    %cd kde_ontology_slm_lab")
    print("Then re-run this cell. We will not auto-clone for you (lab policy:")
    print("recipes only, no auto-downloads).")


## 1. Load the training config

We use the same YAML loader the CLI uses (`src/common/config.py`). The file lives at `configs/training.yaml`.


In [ ]:
from src.common.config import load_yaml
from src.common.paths import CONFIGS

cfg = load_yaml(CONFIGS / 'training.yaml')
models = load_yaml(CONFIGS / 'models.yaml').get('models', {}) or {}
print('top-level keys:', sorted(cfg.keys()))
print('available profiles:', sorted((cfg.get('profiles') or {}).keys()))
print('available model_keys:', sorted(models.keys()))


## 2. Resolve the `colab_t4` profile

Resolution mirrors `src/cli/train_cmd.py`: take the top-level defaults, deep-merge the profile's overlay, attach the chosen model entry.


In [ ]:
def deep_merge(base: dict, overlay: dict) -> dict:
    out = dict(base)
    for k, v in overlay.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = deep_merge(out[k], v)
        else:
            out[k] = v
    return out

profile = 'colab_t4'
model_key = cfg.get('model_key', 'qwen_small')
overlay = (cfg.get('profiles') or {}).get(profile, {})
resolved = deep_merge({k: v for k, v in cfg.items() if k != 'profiles'}, overlay)
resolved['profile'] = profile
resolved['model_key'] = model_key
resolved['resolved_model_entry'] = models.get(model_key, {})


## 3. Print the resolved config


In [ ]:
import json

print(json.dumps(resolved, indent=2, sort_keys=True))


## 4. The launch command (dry-run only)

A real training run would look like the command below. We do not execute it; we do not even import `torch`. To go live the learner must:

1. Set a real `base_model` id in `configs/models.yaml` (replace the `CHANGE_ME`).
2. Install the `[train]` extra (torch, peft, trl, bitsandbytes).
3. Confirm the weights are cached locally.
4. Implement the actual loop in `src/training/` (currently scaffolded).


In [ ]:
model_entry = resolved['resolved_model_entry']
base_model = model_entry.get('base_model', '<set base_model in configs/models.yaml>')
cmd = [
    'accelerate launch',
    '--mixed_precision=fp16',
    '--num_processes=1',
    'src/training/train_sft.py',
    f"--model_name_or_path '{base_model}'",
    f"--dataset_path '{resolved['dataset_path']}'",
    f"--output_dir '{resolved['output_dir']}'",
    f"--max_seq_length {resolved['max_seq_length']}",
    f"--learning_rate {resolved['optim']['learning_rate']}",
    f"--per_device_train_batch_size {resolved['optim']['batch_size']}",
    f"--gradient_accumulation_steps {resolved['optim']['gradient_accumulation_steps']}",
    f"--num_train_epochs {resolved['optim']['num_train_epochs']}",
    f"--lr_scheduler_type {resolved['optim']['lr_scheduler_type']}",
    f"--warmup_steps {resolved['optim']['warmup_steps']}",
    f"--logging_steps {resolved['optim']['logging_steps']}",
    f"--save_steps {resolved['optim']['save_steps']}",
    f"--lora_r {resolved['lora']['r']}",
    f"--lora_alpha {resolved['lora']['alpha']}",
    f"--lora_dropout {resolved['lora']['dropout']}",
    f"--lora_target_modules {resolved['lora']['target_modules']}",
    '--load_in_4bit' if resolved.get('load_in_4bit') else '',
    '--train_on_responses_only' if resolved.get('train_on_responses_only') else '',
    f"--seed {resolved['optim']['seed']}",
]
cmd = [c for c in cmd if c]
print('# Would launch (dry-run only; not executed):')
print(' \\\n    '.join(cmd))


## 5. What a learner does next

- Run `kde-lab train --profile colab_t4` from the CLI — same output as above.
- Diff `colab_t4` against `local_24gb` to see how a bigger GPU changes hyperparameters.
- Read `docs/07_finetune_recipe.md` (companion to this notebook) before flipping the switch from dry-run to live.


## Summary

You inspected the resolved training recipe for Colab T4 and printed the launch command a real run would invoke. Notebook 08 measures the model that recipe would produce.


## Exercises

1. Switch the profile to `local_24gb` and diff the JSON output. What changed and why?
2. Estimate the effective batch size (`batch_size * grad_accum`) for each profile.
3. Replace `CHANGE_ME` in one entry of `configs/models.yaml` with a real HF id (do not commit). Re-run and confirm the launch command picks it up.
